# 04 · Gold – Modelo dimensional (Fabric)

Monta o modelo estrela que alimenta o modelo semântico e o app.

| Tabela | Grão | Chave |
|---|---|---|
| `gold_fato_municipio_ano` | 1 linha por município × ano | (`cod_municipio`, `ano`) |
| `gold_dim_municipio` | 1 linha por município | `cod_municipio` (código IBGE, estável) |
| `gold_dim_tempo` | 1 linha por ano | `ano` |

- **Só os anos presentes nas duas fontes** (premissa 10).
- **Sem PIB per capita gravado na fato**: razão se calcula na medida (soma do PIB ÷ soma da população), senão a agregação por UF ou região fica errada.
- `par_completo` marca as linhas com população **e** PIB, base do PIB per capita.
- **Portão de qualidade**: só roda se a última execução do 03_qualidade não teve FAIL.

In [ ]:
from pyspark.sql import functions as F

# Portão de qualidade: a última execução do 03_qualidade não pode ter FAIL
qa = spark.table("qualidade_resultados")
ultima = qa.agg(F.max("executado_em")).first()[0]
falhas = qa.filter((F.col("executado_em") == ultima) & (F.col("status") == "FAIL")).count()
print(f"Última validação: {ultima} | FAIL: {falhas}")
assert falhas == 0, "A última validação teve FAIL: rode e corrija o 03_qualidade antes da Gold."

pop = spark.table("silver_populacao")
pib = spark.table("silver_pib")
mun = spark.table("silver_municipio")

def gravar(df, tabela):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

In [ ]:
# Anos presentes nas duas fontes (premissa 10)
anos_pop = {r.ano for r in pop.select("ano").distinct().collect()}
anos_pib = {r.ano for r in pib.select("ano").distinct().collect()}
ANOS = sorted(anos_pop & anos_pib)
print(f"{len(ANOS)} anos: {ANOS}")

# Dimensão de tempo
dim_tempo = (spark.createDataFrame([(a,) for a in ANOS], ["ano"])
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("decada", F.concat((F.floor(F.col("ano") / 10) * 10).cast("int").cast("string"), F.lit("s")))
    .withColumn("ultimo_ano", F.col("ano") == max(ANOS)))
gravar(dim_tempo, "gold_dim_tempo")

# Dimensão de município
dim_municipio = mun.select(
    "cod_municipio", "nome_municipio",
    F.concat_ws(" - ", "nome_municipio", "sigla_uf").alias("municipio_uf"),
    "cod_uf", "sigla_uf", "nome_uf", "cod_regiao", "nome_regiao",
)
gravar(dim_municipio, "gold_dim_municipio")

In [ ]:
# Fato: município × ano, só anos em comum, só linhas com pelo menos um valor
p = pop.filter(F.col("ano").isin(ANOS)).select("cod_municipio", "ano", "populacao")
b = pib.filter(F.col("ano").isin(ANOS)).select("cod_municipio", "ano", "pib_reais", "pib_mil_reais")

fato = (p.join(b, ["cod_municipio", "ano"], "full")
         .filter(F.col("populacao").isNotNull() | F.col("pib_reais").isNotNull())
         .withColumn("par_completo", F.col("populacao").isNotNull() & F.col("pib_reais").isNotNull())
         .withColumn("pib_negativo", F.coalesce(F.col("pib_reais") < 0, F.lit(False)))
         .select("cod_municipio", "ano", "populacao", "pib_reais", "pib_mil_reais",
                 "par_completo", "pib_negativo"))
gravar(fato, "gold_fato_municipio_ano")

In [ ]:
# Conferências da Gold
f = spark.table("gold_fato_municipio_ano")
dm = spark.table("gold_dim_municipio")
dt = spark.table("gold_dim_tempo")

dup = f.groupBy("cod_municipio", "ano").count().filter("count > 1").count()
orfao_mun = f.join(dm, "cod_municipio", "left_anti").count()
orfao_ano = f.join(dt, "ano", "left_anti").count()
print(f"Chave duplicada na fato: {dup} | sem município na dimensão: {orfao_mun} | sem ano na dimensão: {orfao_ano}")
assert dup == 0 and orfao_mun == 0 and orfao_ano == 0, "Problema de chave ou integridade na Gold."

# Reconciliação Silver × Gold nos anos usados
s_pop = pop.filter(F.col("ano").isin(ANOS)).agg(F.sum("populacao")).first()[0]
s_pib = pib.filter(F.col("ano").isin(ANOS)).agg(F.sum("pib_reais")).first()[0]
g = f.agg(F.sum("populacao").alias("pop"), F.sum("pib_reais").alias("pib")).first()
print(f"População  Silver = {s_pop:,} | Gold = {g.pop:,}")
print(f"PIB (R$)   Silver = {s_pib:,.0f} | Gold = {g.pib:,.0f}")
assert s_pop == g.pop and s_pib == g.pib, "Totais da Gold diferentes da Silver."

f.groupBy("par_completo").count().show()
# PIB per capita do Brasil por ano, só com linhas de par completo (mesma regra da medida DAX)
(f.filter("par_completo").groupBy("ano")
   .agg(F.count("*").alias("municipios"),
        F.round(F.sum("pib_reais") / F.sum("populacao"), 2).alias("pib_per_capita_brasil"))
   .orderBy("ano").show(30))